# 따라 하기 — 동적 페이지는 Selenium으로
같은 명언 사이트지만 **자바스크립트가 나중에 내용을 채우는** 버전 `quotes.toscrape.com/js/`

requests로는 빈 틀만 오고, Selenium은 진짜 브라우저를 열어 다 그려진 화면에서 모아요.

In [1]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

실행 위치: 내 컴퓨터 (Windows)
GPU: NVIDIA GeForce RTX 5070 Laptop GPU, 8151 MiB
한글 글꼴: ['Malgun Gothic']


In [2]:
# ▶ Selenium 준비 — Colab은 크롬까지 설치(1~2분), 내 컴퓨터는 크롬이 깔려 있으면 OK
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'selenium'])
if IN_COLAB:
    subprocess.run('wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb'
                   ' && apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null', shell=True)
import selenium
print('selenium', selenium.__version__, '준비 완료')

selenium 4.52.0 준비 완료


## ① requests로 받아 보면? — 0개

In [3]:
import requests, time, pandas as pd
from bs4 import BeautifulSoup
BASE = 'https://quotes.toscrape.com'

soup = BeautifulSoup(requests.get(f'{BASE}/js/').text, 'html.parser')
print('div.quote 개수:', len(soup.select('div.quote')))   # 0 → 자바스크립트가 나중에 채우는 페이지

div.quote 개수: 0


## ①-2 Ctrl+U로 보면 — 데이터가 `<script>` 안에 숨어 있다
`/js/` 페이지 소스(Ctrl+U)를 보면 `var data = [ … ]` 안에 명언·저자가 그대로 들어 있어요.
브라우저 없이 그 부분만 잘라 내서 JSON으로 읽을 수 있어요 (`re` = 글자 패턴 찾기, `json` = JSON 읽기).

In [4]:
import re, json
html = requests.get(f'{BASE}/js/').text
m = re.search(r'var data = (\[.*?\]);', html, re.S)     # var data = [ … ]; 부분 잘라 내기
data = json.loads(m.group(1))                              # 글자 → 파이썬 리스트
print('script 안의 명언 수:', len(data))
pd.DataFrame([{'명언': q['text'], '저자': q['author']['name']} for q in data]).head()

script 안의 명언 수: 10


,명언,저자
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


## ② Selenium으로 브라우저를 열어 모으기
선택자는 BeautifulSoup와 **똑같아요** (`div.quote`, `span.text`, `small.author`)

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

opt = webdriver.ChromeOptions()
# opt.add_argument('--headless=new')   # 창 없이 실행하려면 맨 앞 # 지우기 (시연은 창이 보이게 그대로)          # 창 없이 실행 · 내 PC에서 창을 보고 싶으면 이 줄 지우기
opt.add_argument('--no-sandbox')
opt.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome(options=opt)      # 크롬 드라이버는 Selenium이 알아서 맞춰요

driver.get(f'{BASE}/js/')
WebDriverWait(driver, 10).until(            # 명언 상자가 그려질 때까지 최대 10초 기다리기
    EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote')))

rows = []
for box in driver.find_elements(By.CSS_SELECTOR, 'div.quote'):
    rows.append({'명언': box.find_element(By.CSS_SELECTOR, 'span.text').text,
                 '저자': box.find_element(By.CSS_SELECTOR, 'small.author').text})
pd.DataFrame(rows).head()

,명언,저자
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


## ③ 여러 페이지 돌고 CSV로 저장 → 브라우저 닫기

In [6]:
rows = []
for page in range(1, 6):                    # 1~5페이지
    driver.get(f'{BASE}/js/page/{page}/')
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote')))
    for box in driver.find_elements(By.CSS_SELECTOR, 'div.quote'):
        rows.append({'페이지': page,
                     '저자': box.find_element(By.CSS_SELECTOR, 'small.author').text})
    time.sleep(1)
driver.quit()                               # 다 쓰면 꼭 닫기

df = pd.DataFrame(rows)
df.to_csv('quotes_js.csv', index=False, encoding='utf-8-sig')
df['저자'].value_counts().head()

저자
Albert Einstein    8
J.K. Rowling       6
Marilyn Monroe     6
Bob Marley         3
Dr. Seuss          3
Name: count, dtype: int64

## ④ (더 쉬운 길) Network 탭에서 JSON 주소 찾기
`quotes.toscrape.com/scroll` 을 열고 F12 → Network → Fetch/XHR → 스크롤하면 `api/quotes?page=…` 요청이 보여요. 브라우저 없이 바로 받을 수 있어요.

In [7]:
data = requests.get(f'{BASE}/api/quotes?page=1').json()
print(data.keys())
pd.DataFrame([{'명언': q['text'], '저자': q['author']['name']} for q in data['quotes']]).head()

dict_keys(['has_next', 'page', 'quotes', 'tag', 'top_ten_tags'])


,명언,저자
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


**정리:** 먼저 Ctrl+U로 정적/동적 확인 → script 안에 있으면 ①-2(re + json) → 따로 불러오면 ④ Network 탭 JSON → 그래도 없으면 Selenium